In [ ]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer, util

embedder = SentenceTransformer("all-MiniLM-L6-v2")

msgs = [
    "Please send me the OTP you just received",
    "Brother forward me the verification code that came to your phone",
    "Please bro copy the number behind your ATM card and send it",
    "Are we still meeting at the library at 4pm?",
]

emb = embedder.encode(msgs)
print(util.cos_sim(emb, emb).numpy().round(2))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[[1.   0.31 0.38 0.17]
 [0.31 1.   0.5  0.09]
 [0.38 0.5  1.   0.09]
 [0.17 0.09 0.09 1.  ]]


In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()
fname = list(uploaded.keys())[0]

raw = pd.read_csv(fname, encoding="latin-1")
raw = raw.drop_duplicates(subset="TEXT")
raw["LABEL"] = raw["LABEL"].str.lower()

sizes = {"ham": 2000, "spam": 1000, "smishing": 1000}
parts = []
for lab, n in sizes.items():
    pool = raw[raw["LABEL"] == lab]
    parts.append(pool.sample(min(n, len(pool)), random_state=42))

base = pd.concat(parts).rename(columns={"TEXT": "message"})
base["label"] = base["LABEL"].map({"ham": "ham", "spam": "scam", "smishing": "scam"})
base = base[["message", "label"]].reset_index(drop=True)
base.to_csv("base_reduced.csv", index=False)

print(base.shape)
print(base["label"].value_counts())


Saving Dataset_10191 (1).csv to Dataset_10191 (1).csv
(4000, 2)
label
ham     2000
scam    2000
Name: count, dtype: int64


In [ ]:
import pandas as pd

ng_scams = [
    "Your GTBank token has expired. To renew, kindly reply with your 10-digit account number and the 6-digit code sent to you.",
    "Hello, I am calling from First Bank fraud team. Read out the code we just sent so we can block the fraudster.",
    "Abeg oga, my phone dey hang, forward me the OTP wey enter your phone make I confirm say na you.",
    "FCMB alert: your card has been restricted. Tap here and enter your PIN to unrestrict it: http://fcmb-unlock.info",
    "Congratulations, you've been picked for the Dangote youth grant of N750,000. Send your BVN and phone number to register.",
    "Kindly send the 4 digits at the back of your card, we want to process your refund now.",
    "Dear Sir/Madam, your Ecobank account will be closed unless you verify your BVN within 24 hours.",
    "Hi, it's your pastor. I'm stranded, please buy N20,000 recharge card and send me the PIN numbers.",
    "Please I mistakenly entered your number for the transfer. Send me the alert code so the bank can reverse it.",
    "UBA Alert: a login attempt from a new device. If this wasn't you, confirm your PIN and OTP at the link below.",
    "Earn N15,000 daily by liking videos. Pay a N3,000 activation fee to start and send your bank login.",
    "Your NIN slip is ready for collection. Send N2,000 and your NIN to get it delivered to you today.",
    "Your Glo 100GB bonus is waiting. Click the link and log in with your phone number and PIN.",
    "I'm a soldier in Syria with 4.5 million dollars. I need a trusted person to receive it. Send your bank details.",
    "Your Binance account is locked. Send the 6-digit Google Authenticator code to unlock it.",
    "Dear customer, your electricity meter will be disconnected today. Pay the arrears via this link and enter your card PIN.",
    "I am hiring a house help abroad. Send N25,000 for visa processing and your passport number.",
    "Hello dear, I'm new on WhatsApp. Please send me the 6 digit code you'll get so I can verify my account.",
    "Your POS machine licence has expired. Pay N5,000 to this account to avoid being blocked, and send your agent PIN.",
    "Sir, your tax refund of N120,000 is ready. Send your account number, BVN, and card PIN to receive it.",
    "Please bro help me, my bank app locked. Use your own card to receive my money and send me the OTP that comes.",
    "Your JAMB result is being withheld. Pay N4,500 and send your email password for verification.",
    "Quick loan of up to N500,000 with no collateral. Send your BVN and ATM card number to get approved in minutes.",
    "Congrats! Your phone number won the MTN Pulse promo. Send N1,000 airtime to claim your prize of N2,000,000.",
    "I have blocked your SIM for suspicious activity. To unblock, reply with your full name, NIN, and the code we sent.",
    "Hi dear, I love your pictures. I want to send you a gift, kindly send your account number and card details.",
    "FRSC notice: you have an unpaid fine. Pay now at frsc-pay-online.com with your card number and PIN to avoid arrest.",
    "Sorry for disturbing you. I need the code that was just sent to your number, it was sent by mistake.",
    "Moniepoint agent here, your account has a problem. Share your PIN for me to fix it.",
    "Your Amazon order of N280,000 has been placed. If not you, call this number and give the code to cancel.",
    "Your Chipper Cash wallet is on hold. Confirm your login password and BVN to release your funds.",
    "Pay N10,000 to join our forex group and triple your money in a week. Send your account details.",
    "Dear Customer, update your KYC now or your account will be frozen. Send your BVN, NIN and card number.",
    "Mama, I'm at the police station. Please send N40,000 to this account right now and don't tell dad.",
    "Last chance: your inheritance of $3.2 million is ready. Send the processing fee and your account number.",
    "Kindly confirm your ATM card number and expiry date to receive your COVID palliative.",
    "Verification needed: forward the SMS code you will receive from Paystack to complete your refund.",
    "Hello, your DSTV has been upgraded free. Enter your card number and PIN to activate it.",
    "Good day, I'm from Access Bank. A transfer of N300,000 is pending. Give me your OTP to complete it.",
    "Download this app to get your free N50,000 loan. Allow all permissions and enter your BVN.",
    "Your Instagram has been selected for verification. Send your password and the code to get the blue badge.",
    "Abeg help me, I want to collect money from my oga. Send me your card number make I use am.",
    "Your child's school fee payment failed. Re-enter your card details and PIN at the link below.",
    "Your Facebook page will be deleted. Send the 6 digit code we send to you in 10 minutes to appeal.",
    "Win a brand new car! Pay N2,500 for your entry and send your account number and BVN.",
    "Customer care: your wallet was credited by mistake. Please return N45,000 and send the OTP so we can confirm.",
    "Your NNPC promo: free fuel for 6 months. Register with your BVN and card PIN now.",
    "Your account has been selected for a N1m SME grant. Pay N5,000 to unlock and send your BVN.",
    "This is the bank. Your account was hacked. Quickly send your PIN and the code to secure your money.",
    "Please read the 6 digit code on your screen to me, I need it to cancel the wrong transfer.",
]

ng_normals = [
    "Good evening ma, I've sent the report to your email.",
    "Your OTP is 530921. It is valid for 5 minutes. Never share it with anyone, including our staff.",
    "Credit Alert: N25,000.00 from Chidi Okeke on 17-Nov. Available balance: N61,200.45.",
    "Debit Alert: N3,200.00 at SHOPRITE AKURE. Not you? Call our hotline on the back of your card.",
    "Please never share your BVN or PIN with anyone who calls you, even if they say they are from the bank.",
    "I'll be at the bank by 10am to update my BVN, are you coming along?",
    "Network was bad so my OTP didn't come. I'll try again after lunch.",
    "How far na? Did you submit the assignment before the deadline?",
    "The lecturer said the test holds on Tuesday, not Monday.",
    "Mummy has sent the money for my school fees, I'm going to pay today.",
    "Kindly confirm if you received the invoice I sent yesterday.",
    "Your MTN data balance is 1.2GB. Expires on 25 Nov. Dial *131# to renew.",
    "I forgot my ATM PIN again, so annoying. I'll reset it at the branch tomorrow.",
    "Your transaction of N15,000 to Mr Adeyemi was successful.",
    "Happy Sunday! Church starts at 9am today.",
    "Can you send me the link to the Python course you were talking about?",
    "I'm in traffic, I'll be 20 minutes late.",
    "Your NEPA token purchase of N5,000 was successful. Token: 4567-8901-2345-6789-0123.",
    "Please remember to bring your NIN slip for the registration tomorrow.",
    "Dear customer, we have received your request. A representative will contact you within 48 hours.",
    "I got a message asking for my card PIN, I didn't reply. Please warn others.",
    "The code you asked for is in the file I sent, just run the script.",
    "Our hackathon starts on Friday. Please bring your laptop and charger.",
    "Your Bolt ride is arriving. Driver: Samuel, Toyota Corolla, blue.",
    "Can you lend me N2,000 till Friday? I'll pay back when my allowance comes.",
    "My password manager finally works, I stopped reusing the same password everywhere.",
    "Your verification code is 482-193. Enter it on the website to finish signing up.",
    "Congratulations on your promotion! You deserve it.",
    "Please send me the hostel fee receipt, I'll upload it for the office.",
    "The bank manager said my new ATM card will be ready by Thursday.",
    "Jumia: your package is out for delivery. Please keep your phone close.",
    "Don't forget the group meeting by 5pm in the faculty hall.",
    "I'm writing my weather model report, the rainfall forecast is finally working.",
    "Lol, someone tried to scam me yesterday, they asked for my OTP. I blocked them.",
    "Your electricity bill for October is N8,400. Please pay before the 25th.",
    "Dear applicant, your interview is scheduled for Monday 9am. Bring your original certificates. No payment is required.",
    "Pls send the pictures of the notes, I missed the class.",
    "Your Netflix subscription was renewed successfully for N4,400.",
    "Thanks for the transfer, I've received the alert.",
    "The fuel price has gone up again, I'll take the bus tomorrow.",
    "Come over later, mummy cooked jollof.",
    "Your Paystack receipt: N6,000 paid to Akure Tech Hub. Thank you.",
    "Kindly review the pull request when you are free.",
    "Reset your password using the link we sent to your registered email. If you didn't ask for this, ignore it.",
    "Is it true that the bank will charge N50 stamp duty on all transfers above N10,000?",
    "I deposited the cash at the POS stand near the gate, the agent was very helpful.",
    "WAEC timetable has been released, check the website.",
    "Our flight got delayed by 2 hours, I'll call you when I land.",
    "I've updated the Streamlit app, please test it and tell me what you think.",
    "Your appointment with Dr. Musa is confirmed for Wednesday at 11am.",
]

nigerian_train = pd.DataFrame({
    "message": ng_scams + ng_normals,
    "label": ["scam"] * len(ng_scams) + ["ham"] * len(ng_normals),
})
nigerian_train.to_csv("nigerian_train.csv", index=False)

print(nigerian_train.shape)
print(nigerian_train["label"].value_counts())

(100, 2)
label
scam    50
ham     50
Name: count, dtype: int64


In [ ]:
edu_scams = [
    "Congratulations! You have been selected for the 2026 Federal Government Scholarship. Pay N20,000 into this account to confirm your selection.",
    "You have been awarded a full scholarship even without applying. Click the link below to check whether you are among the top 50: http://scholar-check-ng.com",
    "Dear aspirant, we noticed you have been searching for admission. A new university is offering you direct entry. Pay N35,000 acceptance fee to secure your slot.",
    "Your admission is ready! Log in with your JAMB profile email and password at the link to download your admission letter.",
    "Scholarship board: you are 1 of 50 winners. Send your BVN and account number to receive your N300,000 award.",
    "Final notice: your scholarship slot expires in 24 hours. Pay the N15,000 verification fee now or it will be given to another candidate.",
    "Guaranteed admission without JAMB or post-UTME. Pay N150,000 to our agent and get your letter in 3 days.",
    "Abeg your admission don come o. Send N10,000 and your passport photo to this number make dem process am.",
    "UK fully funded scholarship! Send N45,000 for visa processing and your international passport to this WhatsApp number.",
    "Student loan alert: your loan has been approved. Pay N5,000 activation fee and send your NIN and BVN to receive it.",
    "We reviewed your records and you qualify for the Shell scholarship. Reply with your 6 digit code and your card PIN to release the funds.",
    "Your JAMB result has been upgraded to 290. Pay N6,000 to this account to confirm the new score.",
    "Hello student, you have been shortlisted for a free laptop and N100,000 tuition grant. Click to confirm your name on the list.",
    "Admission alert: you have been admitted to study Medicine. Send N200,000 as acceptance fee to this account within 48 hours.",
    "Dear candidate, your scholarship disbursement is pending. Confirm your ATM card number and OTP to receive N250,000.",
    "Canada scholarship is open to Nigerians with no IELTS. Pay N60,000 for a guaranteed offer letter and send your email password so we can apply for you.",
    "Congratulations dear student, you are the lucky winner of our JAMB mock scholarship draw. Reply with your full name, address and card number.",
    "Your name is on the final list! To avoid disqualification, send N8,500 for biometrics and the 6-digit code we just texted you.",
    "We are a new private university. You have been pre-admitted without writing any exam. Click the link to pay your acceptance fee.",
    "The scholarship committee has chosen you. Do not tell anyone. Send N12,000 to confirm your identity and keep your slot.",
    "NELFUND notice: you have been approved but your account is on hold. Send your login password and BVN to release it.",
    "Aunty, I work in the admissions office. For N80,000 I can change your course to Law. Send the money to my personal account.",
    "Your WAEC scholarship is ready for collection. Pay N3,500 clearance fee and send your phone's verification code.",
    "Hello, we saw your application for admission three years ago. A new opportunity is here. Pay N25,000 to open your portal.",
    "Chevening scholarship alert: you are selected! Send N30,000 to the processing agent and your passport scan today.",
    "Dear parent, your child has been awarded a N500,000 school-fee grant. Send your card number and PIN to receive it.",
    "Final reminder: confirm your scholarship by paying N18,000 or your slot will be reassigned within the hour.",
    "You don win scholarship o! Click here, put your phone number and PIN to see your name on the list.",
    "Admission list is out. Your name is there but you must pay N7,000 through this link before it disappears.",
    "Your scholarship application was successful. Forward the OTP sent to your phone so we can credit your account.",
]

edu_normals = [
    "Congratulations! You have been offered admission to study Meteorology at FUTA. Visit the admissions office with your original documents. No payment is needed to collect the letter.",
    "Your JAMB UTME exam is on Saturday at 9am. Print your exam slip from your profile and arrive early.",
    "School fees for 2026/2027 are N75,000, payable through Remita on the official school portal only.",
    "Did you see the admission list? I saw my name!",
    "The scholarship application closes on Friday. Apply on the official website. We never ask applicants for payment.",
    "Your scholarship stipend of N30,000 has been credited to your account.",
    "Reminder: your course registration deadline is this Friday.",
    "Chevening applications open next month, I'm preparing my essays.",
    "Your post-UTME screening is scheduled for Thursday. Bring your slip and ID card.",
    "Please remember that nobody can give you admission for money. Apply only through the official portal.",
    "Dear student, your hostel allocation is ready. Check the portal for your room number.",
    "I applied for three scholarships this week, fingers crossed.",
    "The faculty office said results will be released by next week.",
    "Your transcript request has been received. It will be ready in 5 working days.",
    "Our department is organising a free coding workshop on Saturday at 10am in the ICT hall. All are welcome.",
    "Don't pay anyone for admission o, that's how people lose money. Use the school website.",
    "Dear parent, school resumes on Monday, 3 January. Please ensure your child has all textbooks.",
    "I got the interview invitation, it's on Wednesday at the university's main campus.",
    "JAMB reminder: never share your profile password with anyone.",
    "Registration for the NYSC orientation camp has started. Use the official NYSC portal.",
    "The lecturer has uploaded the slides for the Climate Physics course.",
    "Your library card has been renewed for another year.",
    "Please help me check the cut-off mark for Computer Science this year.",
    "The exam timetable is out, check the department notice board.",
    "I just finished my scholarship interview, it went well.",
    "Congratulations on your scholarship! Remember to keep your grades up.",
    "Student ID cards are ready for collection at the registry from 9am to 4pm.",
    "My brother is looking for admission this year, any advice on where to apply?",
    "Your tuition payment of N75,000 was received. Receipt number: FUT-20481.",
    "A new private university just opened in our state, please verify its NUC accreditation before applying.",
]

edu_df = pd.DataFrame({
    "message": edu_scams + edu_normals,
    "label": ["scam"] * len(edu_scams) + ["ham"] * len(edu_normals),
})

nigerian_train = pd.concat([nigerian_train, edu_df], ignore_index=True).drop_duplicates(subset="message")
nigerian_train.to_csv("nigerian_train.csv", index=False)

print(nigerian_train.shape)
print(nigerian_train["label"].value_counts())

(160, 2)
label
scam    80
ham     80
Name: count, dtype: int64


In [ ]:
import pandas as pd

# ---------- First 60 messages (TRAINING) ----------
first_scams = [
    "Dear customer, your account will be blocked today. Reply with your BVN and ATM PIN to keep it active.",
    "GTBank alert: unusual login detected. Confirm your card number and OTP at http://gtb-secure-verify.com to avoid suspension.",
    "Hello, I mistakenly sent money to your account. Please send back the code that was sent to your phone so I can reverse it.",
    "Congratulations! You have been selected for a N500,000 grant. Send your account number and BVN to receive it.",
    "Please read me the 6 digit number you just received, I need it to confirm something for you.",
    "Your MTN data promo is ready. Click here to claim 50GB free and enter your password.",
    "Urgent: your debit card has been deactivated. Call this number now and give the agent the 3 digits at the back of your card.",
    "Hi dear, I am stuck at the airport and need money urgently. Send me your card details and I will pay you back double.",
    "Invest N20,000 today and receive N200,000 in 24 hours. Send your bank details now to join.",
    "We are hiring remote workers. Pay a N5,000 registration fee and send your login password to start today.",
    "Your account has been suspended. Verify your identity immediately at the link below or it will be closed.",
    "Final notice: your NIN is about to expire. Send your NIN and BVN to this WhatsApp number to avoid being blocked.",
    "Brother please forward me the verification code that just came to your phone, my own is not working.",
    "Dear user, your Access Bank token has expired. Enter your PIN here to renew: http://bit.ly/renew-token",
    "You won the Airtel lucky draw! Send N2,000 to claim your new iPhone.",
    "Mummy it's me, I lost my phone. Please send N30,000 to this account urgently, don't call me.",
    "Your PayPal account is limited. Log in with your email and password at paypal-secure-help.com to restore access.",
    "Copy the numbers on the front and back of your card and send them to me so I can pay you faster.",
    "Bitcoin giveaway: send 0.1 BTC to this address and get 1 BTC back instantly.",
    "CBN is giving free money to citizens. Register with your BVN, PIN and phone number now.",
    "Dear student, your scholarship is approved. Send your school portal password to claim it.",
    "Your package is held at customs. Pay N3,500 delivery fee at this link to release it.",
    "Please send me your Opay PIN so I can help you unlock your account, it was flagged.",
    "Hi, tell me the code Google just sent you, I'm trying to verify that it's really you.",
    "Your electricity token purchase failed. Re-enter your card number, expiry date and CVV to complete it.",
    "Congrats winner!! Reply with your full name, address and card number to receive your free gift.",
    "Your WhatsApp will be deleted in 24 hours. Click this link and enter the code we send you to keep it.",
    "Make N100k weekly from home. Send me your bank login so I can set up your earnings.",
    "I am a prince with 5 million dollars to transfer, I need your account details and a small fee to start.",
    "Security alert: someone tried to use your card. Reply with the OTP you just received to cancel the transaction.",
]

first_normals = [
    "Hey, are we still meeting at the library at 4pm?",
    "Please send me the lecture notes when you get a chance, thanks.",
    "Your OTP is 482913. It expires in 5 minutes. Do not share this code with anyone.",
    "Debit alert: N5,000 has been withdrawn from your account at ATM. If this was not you, visit your nearest branch.",
    "I just checked my bank app, the transfer has arrived. Thanks so much!",
    "Don't forget to pin the meeting location on the group chat.",
    "Can you help me with the Python assignment? I'm stuck on loops.",
    "Mum, I've arrived safely. I'll call you tonight.",
    "Our class is moved to Thursday at 10am in Room 204.",
    "Please bring your student ID card to the exam hall tomorrow.",
    "Happy birthday! Wishing you all the best, enjoy your day.",
    "Credit alert: N10,000 received from Tunde. Balance is N24,350.",
    "I forgot my password for the school portal, how do I reset it?",
    "Let's meet at the mall at 6. Bring your money so we can split the bill.",
    "The weather forecast says heavy rain tomorrow, carry an umbrella.",
    "Abeg send me the assignment, I never see am.",
    "How far? You dey come church later?",
    "Your Uber is arriving in 3 minutes. Your driver is in a white Toyota.",
    "Reminder: your dentist appointment is tomorrow at 2pm.",
    "I changed my ATM card at the bank today, the new one will be ready next week.",
    "Thanks for the call. Send the project files to my email when you can.",
    "Your order has been shipped and will arrive on Friday.",
    "Your verification code is 731902. If you didn't request this, please ignore this message.",
    "Can you share the slides from today's tech meetup?",
    "We are hiring a junior developer. Send your CV to the email on our careers page, no payment is required.",
    "Did you see the match last night? That goal was crazy.",
    "Please remember to back up your files before updating your laptop.",
    "I'll send the money for the tickets tomorrow morning, is that fine?",
    "Dr. Bello wants everyone to submit the report by Monday.",
    "Your electricity token is 1234-5678-9012-3456-7890. Units: 25.4 kWh.",
]

train_first60 = pd.DataFrame({
    "message": first_scams + first_normals,
    "label": ["scam"] * len(first_scams) + ["ham"] * len(first_normals),
})

# ---------- Locked test set (80) - DO NOT TUNE ON THIS ----------
test_scams = [
    "Please bro can you help me check my ATM card at home and copy the number behind it",
    "Good afternoon sir, this is your bank. We noticed a strange transaction, kindly confirm your BVN and the code we sent to you.",
    "Your Kuda account has been limited. To restore it, reply with your 4 digit PIN and the OTP.",
    "Hello, I'm from the electricity board, send your card number and expiry date to get your refund.",
    "Sis abeg help me with the code wey dem send to your phone, na for verification.",
    "Dear customer your SIM will be disconnected in 2 hours. Link your NIN now: http://ng-nin-link.xyz",
    "Hi, I sent N50,000 to your account by mistake. Tell me the OTP you got so I can cancel it.",
    "Palmpay promo: win N100,000. Enter your PIN and BVN here to confirm your identity.",
    "Your Moniepoint POS is blocked. Call 0801234567 and give your card PIN to reactivate.",
    "We have a job for you at an oil company. Send N10,000 for processing and your BVN for background check.",
    "Please confirm the 6-digit code you will receive shortly, it's for your loan approval.",
    "FIRS tax refund of N85,000 is waiting. Submit your account number, BVN and ATM PIN.",
    "Your Netflix payment failed. Update your card details now at netflix-billing-help.com.",
    "Dear Sir, kindly urgent: forward me the token sent to your phone so we can finish your transaction.",
    "This is the police. Your account is linked to fraud. Send your BVN and OTP to clear your name.",
    "Congratulations dear, you are our lucky customer of the month. Click to claim and enter your card PIN.",
    "Hi love, I am sending you a gift card. Please send me your card number so I can load the money.",
    "Your Instagram account violates policy. Send the verification code to appeal within 1 hour.",
    "Aunty it's me your nephew. MTN don block my line, abeg send N5,000 airtime to this number now.",
    "Your WAEC result is ready. Pay N2,500 and enter your email password to view.",
    "Your loan of N300,000 is approved. To receive it, pay N15,000 insurance fee first.",
    "Update your bank app now to avoid losing your money: http://bit.ly/bank-update-ng",
    "Please do me a favour and tell me the last 4 digits and the CVV of your debit card, I want to pay your fees.",
    "Your BVN has been flagged. Reply YES with your date of birth and OTP to unflag it.",
    "Double your money! Send N5,000 now and get N50,000 within 1 hour. Send your account details.",
    "Dear customer, your ATM card is expired. Send the card number and PIN to get a new one delivered free.",
    "Hello, this is Opay support. Share the code we just sent to confirm you are the owner.",
    "Your Spotify account has been suspended. Verify your password now.",
    "Sir, your NYSC allowee is ready. Send your account number, BVN and PIN to receive it.",
    "I'm abroad and want to send you money for your business. Just send your bank details and the OTP to receive it.",
    "Urgent! Your DSTV subscription expires today. Enter card details at dstv-renew-now.net.",
    "Please I am in hospital, the bank has blocked my card, send me your PIN so I can pay the bill and I'll refund you.",
    "Your Zenith Bank profile is locked. Unlock it by replying with the token and your login password.",
    "Get free 20GB data now: https://free-data-glo.xyz enter your phone number and PIN.",
    "Your account number has won N1,000,000 in the CBN empowerment draw. Send your BVN to claim.",
    "Kindly send me the OTP, it's for your own good, we are updating your records.",
    "Your Telegram login code is needed to reverse the payment. Send it now.",
    "Bank alert: Your account is on hold. Visit this link and re-enter your ATM card details.",
    "Bro help me, my own card don block. Make I use your card pay small thing, send the number and CVV.",
    "Send your details: full name, BVN, mother's maiden name and card PIN, to receive your COVID relief package.",
]

test_normals = [
    "Good morning sir, please kindly review the attached report.",
    "I forgot my ATM PIN so I'm going to the bank tomorrow to reset it.",
    "Don't share your OTP with anyone, even bank staff. Stay safe.",
    "My BVN is linked to my new account now, the bank confirmed it today.",
    "Your OTP for Kuda login is 482001. Valid for 10 minutes. Never share it.",
    "Credit: N12,500.00 from Ibrahim. Bal: N47,890.20. Thanks for banking with us.",
    "The ATM at the school gate is out of cash again, try the one near the market.",
    "I'm at the bank to open an account, they asked me for my BVN and ID.",
    "Please remind me to pay the NEPA bill before Friday.",
    "Chai, this Python error is stressing me. IndexError again.",
    "Are you coming for the tech meetup on Saturday? Bring your laptop.",
    "Dear student, the portal will be down for maintenance tonight from 11pm.",
    "Thank you for your payment of N3,000. Your receipt is attached.",
    "My sister sent me the money, I'll pay you back tomorrow morning.",
    "How far, you don chop? Make we meet for the cafeteria.",
    "Your Jumia order #4521 has been delivered. Rate your experience.",
    "Please use a strong password with letters, numbers and symbols for your new account.",
    "I need to change my phone number on my bank profile, who knows the process?",
    "Reminder: submit your project proposal by Wednesday 5pm.",
    "Your flight to Abuja is on time. Boarding starts at 6:15pm.",
    "Don't click links from strangers, someone tried to scam me yesterday with a fake bank link.",
    "Withdrawal alert: N20,000 at ATM Akure. Call 0700-BANK if you did not make this.",
    "Can you teach me how to use pandas? I want to clean the student data.",
    "Mama asked me to bring her ATM card from the room, I'll do it now.",
    "Your password was changed successfully. If this wasn't you, contact support.",
    "The lecture notes are in the group, check the pinned message.",
    "Happy new month! May this month bring you good news.",
    "I'll transfer the N5,000 once I get my allowance on Monday.",
    "Your appointment at the clinic is confirmed for 9am tomorrow.",
    "Please share the Zoom link for the meeting.",
    "Network is bad here, I couldn't receive the verification code. Will try again later.",
    "Congratulations on your admission to FUTA! Welcome to the university.",
    "The bank said my new card will be ready in two weeks.",
    "Dear customer, your data bundle of 5GB has been activated and expires in 30 days.",
    "Interview invitation: Junior developer role, Monday 10am via Google Meet. No fees required.",
    "Did you get the OTP? Mine hasn't come yet, I'm trying to log in to the portal.",
    "Make sure you don't give your PIN to anybody, na wetin I tell my mum today.",
    "Your GitHub login code is 224118. It expires in 10 minutes.",
    "Send me your account number, I'll transfer your share of the bill.",
    "Is the tech meetup still holding on Saturday? Please confirm so I can arrange transport.",
]

test_df = pd.DataFrame({
    "message": test_scams + test_normals,
    "label": ["scam"] * len(test_scams) + ["ham"] * len(test_normals),
})

train_first60.to_csv("train_first60.csv", index=False)
test_df.to_csv("locked_test_80.csv", index=False)

print("First 60 (training):", train_first60.shape, train_first60["label"].value_counts().to_dict())
print("Locked test:", test_df.shape, test_df["label"].value_counts().to_dict())

First 60 (training): (60, 2) {'scam': 30, 'ham': 30}
Locked test: (80, 2) {'scam': 40, 'ham': 40}


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

base["source"] = "base"
nigerian_train["source"] = "nigerian"
train_first60["source"] = "first60"

all_df = pd.concat([base, nigerian_train, train_first60], ignore_index=True)
all_df = all_df.drop_duplicates(subset="message").reset_index(drop=True)

strat_key = all_df["label"] + "_" + all_df["source"]
train_df, val_df = train_test_split(
    all_df, test_size=0.2, random_state=42, stratify=strat_key
)

print("Training:", len(train_df), "| Validation:", len(val_df))
print(val_df["source"].value_counts().to_dict())

X_train = embedder.encode(train_df["message"].tolist(), batch_size=64, show_progress_bar=True)
X_val = embedder.encode(val_df["message"].tolist(), batch_size=64)

y_train = (train_df["label"] == "scam").astype(int).values
y_val = (val_df["label"] == "scam").astype(int).values

ng_mask = (val_df["source"] != "base").values

for w in [1, 5]:
    weights = np.where(train_df["source"].values == "base", 1.0, float(w))
    clf = LogisticRegression(max_iter=2000)
    clf.fit(X_train, y_train, sample_weight=weights)
    pred = clf.predict(X_val)
    print(f"\nNigerian-data weight = {w}")
    print("  Overall validation accuracy:        ", round(accuracy_score(y_val, pred), 4))
    print("  Nigerian-style validation accuracy: ", round(accuracy_score(y_val[ng_mask], pred[ng_mask]), 4),
          f"({ng_mask.sum()} messages)")

Training: 3376 | Validation: 844
{'base': 800, 'nigerian': 32, 'first60': 12}


Batches:   0%|          | 0/53 [00:00<?, ?it/s]


Nigerian-data weight = 1
  Overall validation accuracy:         0.9656
  Nigerian-style validation accuracy:  0.7045 (44 messages)

Nigerian-data weight = 5
  Overall validation accuracy:         0.9656
  Nigerian-style validation accuracy:  0.7727 (44 messages)


In [ ]:

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression

emb_df = pd.concat([train_df, val_df], ignore_index=True)
X_all = np.vstack([X_train, X_val])
y_all = (emb_df["label"] == "scam").astype(int).values
is_ng = (emb_df["source"] != "base").values
ng_idx = np.where(is_ng)[0]
base_idx = np.where(~is_ng)[0]
print("Nigerian-style messages:", len(ng_idx))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"\n{'weight':>6} {'C':>4} {'accuracy':>9} {'scam recall':>12} {'normal acc':>11}")
for w in [5, 10, 20, 50]:
    for C in [1, 10]:
        preds = np.zeros(len(emb_df), dtype=int)
        for tr, te in skf.split(ng_idx, y_all[ng_idx]):
            train_idx = np.concatenate([base_idx, ng_idx[tr]])
            weights = np.where(is_ng[train_idx], float(w), 1.0)
            clf = LogisticRegression(max_iter=2000, C=C)
            clf.fit(X_all[train_idx], y_all[train_idx], sample_weight=weights)
            preds[ng_idx[te]] = clf.predict(X_all[ng_idx[te]])
        yt = y_all[ng_idx]
        yp = preds[ng_idx]
        acc = (yt == yp).mean()
        recall = (yp[yt == 1] == 1).mean()
        normal_acc = (yp[yt == 0] == 0).mean()
        print(f"{w:>6} {C:>4} {acc:>9.3f} {recall:>12.3f} {normal_acc:>11.3f}")

Nigerian-style messages: 220

weight    C  accuracy  scam recall  normal acc
     5    1     0.823        0.891       0.755
     5   10     0.814        0.864       0.764
    10    1     0.832        0.900       0.764
    10   10     0.823        0.882       0.764
    20    1     0.832        0.900       0.764
    20   10     0.800        0.873       0.727
    50    1     0.841        0.891       0.791
    50   10     0.809        0.873       0.745


In [ ]:
w, C = 10, 1
probs = np.zeros(len(emb_df))

for tr, te in skf.split(ng_idx, y_all[ng_idx]):
    train_idx = np.concatenate([base_idx, ng_idx[tr]])
    weights = np.where(is_ng[train_idx], float(w), 1.0)
    clf = LogisticRegression(max_iter=2000, C=C)
    clf.fit(X_all[train_idx], y_all[train_idx], sample_weight=weights)
    probs[ng_idx[te]] = clf.predict_proba(X_all[ng_idx[te]])[:, 1]

errs = emb_df.iloc[ng_idx].copy()
errs["scam_prob"] = probs[ng_idx]
errs["pred"] = np.where(errs["scam_prob"] >= 0.5, "scam", "ham")
wrong = errs[errs["pred"] != errs["label"]]

print("=== NORMAL messages wrongly flagged as SCAM ===")
fp = wrong[wrong["label"] == "ham"].sort_values("scam_prob", ascending=False)
for _, r in fp.head(15).iterrows():
    print(f"[{r['scam_prob']:.2f}] {r['message'][:110]}")

print("\n=== SCAMS the model MISSED ===")
fn = wrong[wrong["label"] == "scam"].sort_values("scam_prob")
for _, r in fn.head(15).iterrows():
    print(f"[{r['scam_prob']:.2f}] {r['message'][:110]}")

=== NORMAL messages wrongly flagged as SCAM ===
[0.96] Debit Alert: N3,200.00 at SHOPRITE AKURE. Not you? Call our hotline on the back of your card.
[0.96] Your verification code is 731902. If you didn't request this, please ignore this message.
[0.95] Your transaction of N15,000 to Mr Adeyemi was successful.
[0.93] Your verification code is 482-193. Enter it on the website to finish signing up.
[0.93] Please remember that nobody can give you admission for money. Apply only through the official portal.
[0.89] Reset your password using the link we sent to your registered email. If you didn't ask for this, ignore it.
[0.89] Your NEPA token purchase of N5,000 was successful. Token: 4567-8901-2345-6789-0123.
[0.88] Debit alert: N5,000 has been withdrawn from your account at ATM. If this was not you, visit your nearest branc
[0.80] Credit alert: N10,000 received from Tunde. Balance is N24,350.
[0.79] Please never share your BVN or PIN with anyone who calls you, even if they say they are fro

In [ ]:
more_normals = [
    "GTBank Alert: Acct 0123***89 Debit NGN 8,500.00 on 17-NOV-2026. Desc: POS/ AKURE MARKET. Bal NGN 22,100.50.",
    "Credit Alert! Amt: NGN 50,000.00 from AMINA YUSUF. Bal: NGN 71,540.00. Thank you for banking with Zenith.",
    "Access Bank: Your transfer of N7,000 to Mrs Okoro was successful. Ref: 77213349.",
    "UBA Alert: N2,500.00 airtime purchase successful. Your balance is N18,320.00.",
    "Kuda: You received N15,000 from Tobi. Your new balance is N15,420.",
    "Opay: Your withdrawal of N10,000 was successful. Thanks for using Opay.",
    "Moniepoint: N4,200 payment received from POS terminal. Settlement in 24 hours.",
    "First Bank: A debit of N6,000 occurred on your account. If you did not authorise this, visit any branch or call our official line.",
    "Sterling Bank: Your standing order of N20,000 was processed today.",
    "Palmpay: Bill payment successful. DSTV Compact N10,500. Receipt sent to your email.",
    "Your monthly account statement is ready. Log in to the official mobile app to view it.",
    "Wema Alat: Your account balance is N41,900.12 as of 5:00pm today.",
    "Fidelity Bank: Your new debit card has been dispatched and will arrive in 3 to 5 working days.",
    "Debit Alert: N1,500 for SMS charges. Thank you for banking with us.",
    "Your card ending 4821 was used for N9,800 at JUMIA. If this was you, no action is needed.",
    "Your OTP is 884210. It expires in 10 minutes. Do not share this code with anyone.",
    "Use 392817 as your login code for your Flutterwave account. We will never ask you for this code.",
    "G-482913 is your Google verification code.",
    "Your Paystack verification code is 640218. If you did not request it, please ignore.",
    "Your PalmPay PIN reset code is 193847. Valid for 5 minutes. Never share it.",
    "Your Bolt verification code is 7342.",
    "Use code 551203 to confirm your WhatsApp number. Don't share this code with others.",
    "Your Opay login OTP: 306198. Expires in 5 minutes.",
    "Your transaction OTP is 928374 for the payment of N5,000 to Jumia. Do not disclose it to anyone.",
    "Your Instagram security code is 718293.",
    "Your Binance verification code is 428193. Valid for 30 minutes. Never share this with anyone.",
    "Welcome to Chowdeck! Your sign-in code is 6021.",
    "Your Jumia order #8821 has been shipped. Expected delivery: Thursday.",
    "Thank you for your order. Your payment of N14,500 has been received. Receipt attached.",
    "Your DSTV subscription has been renewed successfully until 17 Dec.",
    "Your Spotify payment of N1,100 was successful.",
    "Your MTN airtime top-up of N1,000 was successful. Balance: N1,240.",
    "Your Airtel data bundle of 3GB was activated. Valid for 30 days.",
    "Your Bolt trip receipt: N2,400, 17 Nov, 8:14am. Thanks for riding.",
    "Your electricity token purchase of N10,000 was successful. Token: 3456-7821-9034-1256-8890.",
    "Your NYSC call-up letter has been uploaded to your portal. Print it from the official website.",
    "Your package has arrived at the pickup station. Bring your delivery code to collect it.",
    "Your bus ticket from Akure to Lagos is confirmed. Departure: 7am tomorrow.",
    "Your Glo bonus data has expired. Dial *127# to buy a new plan.",
    "Beware of scammers. The bank will never ask for your PIN, OTP or password.",
    "Please warn your family: don't send your card details to anyone, even if they call you madam or sir.",
    "Scam alert: fake loan apps are asking for your BVN. Don't fall for them.",
    "I reported the fake admission message to the school. Don't pay anyone.",
    "A guy called me pretending to be from my bank. I told him I'd come to the branch.",
    "NUC reminds the public: admission is only through the official portal. Never pay agents.",
    "Reminder from your bank: never click links in unexpected SMS, use the official app.",
    "I just got a message asking me to send my OTP, I know it's a scam so I deleted it.",
    "Please don't forward your verification code to anyone, not even your best friend.",
    "Our IT team will never ask for your password. Report suspicious emails to the helpdesk.",
    "That Binance giveaway is fake, they are just collecting people's details.",
    "Dear customer, your complaint has been resolved. Thank you for your patience.",
    "Please be informed that the school portal will be unavailable on Saturday for maintenance.",
    "Dear student, your fee payment receipt has been uploaded to your student portal.",
    "Your visa interview is on 5 Dec at 9:30am. Bring the confirmation page printed from the official website.",
    "Hello, your pharmacy order is ready for pickup. We close at 8pm.",
    "Your internet subscription of N15,000 will expire in 3 days. Renew in the official app.",
    "Your flight check-in is now open. Check in online at the airline's official website.",
    "Your results have been released. Log in to the school portal to view them.",
    "Hello, this is a reminder that your car service is due on Friday.",
    "Your delivery rider is outside. Please come to the gate.",
]

more_scams = [
    "Debit Alert: N85,000 transfer to Emeka. If this was not you, reply with your PIN to cancel it now.",
    "Credit Alert: N150,000 pending in your account. Click http://claim-credit-ng.com and enter your card details to receive it.",
    "Your verification code is 482915. Send this code back to us to receive your N20,000 reward.",
    "Wema Alat: Your account has been restricted. Verify at alat-secure-login.net with your PIN and OTP.",
    "Kuda: You received N500,000 by mistake. Return it and forward the OTP we just sent to your phone.",
    "Opay: Your withdrawal of N90,000 is on hold. Reply with your OTP to approve or call this number.",
    "We tried to deliver your package. Pay N1,800 re-delivery fee at this link and enter your card number.",
    "Your electricity token is ready. Pay the N3,000 service fee here: http://nepa-token-pay.info and enter your PIN.",
    "Your Instagram security code is 718293. Forward this code to this number to keep your account.",
    "Your DSTV subscription failed. Click here to update your card number and PIN.",
    "Your Paystack verification code is 640218. Our agent will call you, read it out to confirm your payment.",
    "Your ATM card will be blocked today. Click the link to update your details and enter your PIN.",
    "Your NIN is not linked. Pay N1,500 at this link to avoid your line being barred.",
    "Account alert: someone logged in from Lagos. Click to secure your account and enter your password.",
    "Your admission letter is ready. Pay N5,000 collection fee through this link: http://admission-letter-ng.com",
    "Your bank has upgraded its system. Confirm your account details at the link below to avoid losing access.",
    "Hi, I'm from MTN. To win N100,000, forward the 6-digit code you will receive within 5 minutes.",
    "WhatsApp: your account will expire. Send the code in the message you receive to keep your number active.",
    "Your tax clearance is due. Pay N8,000 via this link and enter your BVN to get your certificate.",
    "Congratulations! You have won the Jumia mega draw. Pay N2,000 delivery fee to claim your phone.",
    "Dear user, your Google account has an issue. Reply with the code Google sends you to fix it.",
    "Our agent is at your gate to deliver your parcel. Give him your card PIN to confirm payment.",
    "Your transfer failed. Click here, log in with your bank details, and retry within 10 minutes.",
    "A package for you is waiting at customs. Send N4,000 and your NIN to release it.",
    "Your Binance withdrawal is pending. Send your verification code to our support to approve it.",
    "Quick! Your SIM will be barred in 1 hour. Click this link and enter your NIN and BVN to avoid it.",
    "Your POS settlement is delayed. Send your PIN to our agent so we can release the N150,000.",
    "Your electricity bill is overdue. Pay N6,500 at this link now or your meter will be removed.",
    "Hello dear, I'm sending you money. Please tell me the code you receive so the transfer can go through.",
    "Your loan repayment is overdue. Pay N12,000 to this account today or we will contact your family.",
    "Fidelity Bank alert: your card was used abroad. Reply with your PIN and OTP to block it.",
    "Verify your Flutterwave account now. Click and enter your login password within 20 minutes.",
    "Your WAEC certificate is ready. Pay N6,000 to this personal account and send your passport photo.",
    "Please bro, a code was sent to your phone by mistake, send it to me quickly.",
    "Your Airtel line has won a car. Send your NIN and N3,000 to confirm your delivery details.",
    "Your Netflix account will be suspended. Update your payment at this link now and enter the OTP.",
    "Our agent will deliver your loan cash. Send the N10,000 insurance fee first, then your BVN.",
    "Confirm your scholarship by clicking this link and entering your JAMB password.",
    "Your DSTV is free for 6 months! Pay N1,000 activation and send your card number.",
    "Dear customer, your account is overdrawn. Send your BVN and PIN to our officer to avoid penalty.",
]

more_df = pd.DataFrame({
    "message": more_scams + more_normals,
    "label": ["scam"] * len(more_scams) + ["ham"] * len(more_normals),
})

nigerian_train = pd.concat([nigerian_train, more_df], ignore_index=True).drop_duplicates(subset="message")
nigerian_train.to_csv("nigerian_train.csv", index=False)

print(nigerian_train.shape)
print(nigerian_train["label"].value_counts())

(260, 3)
label
ham     140
scam    120
Name: count, dtype: int64


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression

all2 = pd.concat([
    base.assign(source="base"),
    nigerian_train.assign(source="nigerian"),
    train_first60.assign(source="first60"),
], ignore_index=True).drop_duplicates(subset="message").reset_index(drop=True)

lookup = dict(zip(emb_df["message"], X_all))
missing = [m for m in all2["message"] if m not in lookup]
print("New messages to embed:", len(missing))
if missing:
    new_emb = embedder.encode(missing, batch_size=64, show_progress_bar=True)
    for m, e in zip(missing, new_emb):
        lookup[m] = e

X2 = np.vstack([lookup[m] for m in all2["message"]])
y2 = (all2["label"] == "scam").astype(int).values
is_ng2 = (all2["source"] != "base").values
ng_idx2 = np.where(is_ng2)[0]
base_idx2 = np.where(~is_ng2)[0]
print("Nigerian-style messages:", len(ng_idx2))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def run_cv(w, C):
    probs = np.zeros(len(all2))
    for tr, te in skf.split(ng_idx2, y2[ng_idx2]):
        train_idx = np.concatenate([base_idx2, ng_idx2[tr]])
        weights = np.where(is_ng2[train_idx], float(w), 1.0)
        clf = LogisticRegression(max_iter=2000, C=C)
        clf.fit(X2[train_idx], y2[train_idx], sample_weight=weights)
        probs[ng_idx2[te]] = clf.predict_proba(X2[ng_idx2[te]])[:, 1]
    return probs

print(f"\n{'weight':>6} {'accuracy':>9} {'scam recall':>12} {'normal acc':>11}")
yt = y2[ng_idx2]
for w in [5, 10, 20, 50]:
    probs = run_cv(w, 1)
    yp = (probs[ng_idx2] >= 0.5).astype(int)
    print(f"{w:>6} {(yt == yp).mean():>9.3f} {(yp[yt == 1] == 1).mean():>12.3f} {(yp[yt == 0] == 0).mean():>11.3f}")

probs = run_cv(10, 1)
errs = all2.iloc[ng_idx2].copy()
errs["scam_prob"] = probs[ng_idx2]
errs["pred"] = np.where(errs["scam_prob"] >= 0.5, "scam", "ham")
wrong = errs[errs["pred"] != errs["label"]]

print("\n=== NORMAL messages wrongly flagged as SCAM (weight 10) ===")
for _, r in wrong[wrong["label"] == "ham"].sort_values("scam_prob", ascending=False).head(12).iterrows():
    print(f"[{r['scam_prob']:.2f}] {r['message'][:110]}")

print("\n=== SCAMS still MISSED (weight 10) ===")
for _, r in wrong[wrong["label"] == "scam"].sort_values("scam_prob").head(12).iterrows():
    print(f"[{r['scam_prob']:.2f}] {r['message'][:110]}")

New messages to embed: 100


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Nigerian-style messages: 320

weight  accuracy  scam recall  normal acc
     5     0.787        0.867       0.718
    10     0.797        0.827       0.771
    20     0.844        0.860       0.829
    50     0.844        0.873       0.818

=== NORMAL messages wrongly flagged as SCAM (weight 10) ===
[0.94] Please remember that nobody can give you admission for money. Apply only through the official portal.
[0.92] Your Glo bonus data has expired. Dial *127# to buy a new plan.
[0.87] Scam alert: fake loan apps are asking for your BVN. Don't fall for them.
[0.87] Reset your password using the link we sent to your registered email. If you didn't ask for this, ignore it.
[0.85] Dear customer, we have received your request. A representative will contact you within 48 hours.
[0.83] Your verification code is 482-193. Enter it on the website to finish signing up.
[0.82] Your verification code is 731902. If you didn't request this, please ignore this message.
[0.80] I just got a message asking m

In [ ]:
from google.colab import files

for f in ["base_reduced.csv", "nigerian_train.csv", "train_first60.csv", "locked_test_80.csv"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from sentence_transformers import SentenceTransformer

embedder2 = SentenceTransformer("all-mpnet-base-v2")
embedder2.max_seq_length = 128

X3 = embedder2.encode(all2["message"].tolist(), batch_size=32, show_progress_bar=True)

def run_cv2(X, w, C):
    probs = np.zeros(len(all2))
    for tr, te in skf.split(ng_idx2, y2[ng_idx2]):
        train_idx = np.concatenate([base_idx2, ng_idx2[tr]])
        weights = np.where(is_ng2[train_idx], float(w), 1.0)
        clf = LogisticRegression(max_iter=2000, C=C)
        clf.fit(X[train_idx], y2[train_idx], sample_weight=weights)
        probs[ng_idx2[te]] = clf.predict_proba(X[ng_idx2[te]])[:, 1]
    return probs

print(f"\n{'weight':>6} {'C':>4} {'accuracy':>9} {'scam recall':>12} {'normal acc':>11}")
yt = y2[ng_idx2]
for w in [10, 20, 50]:
    for C in [1, 10]:
        probs = run_cv2(X3, w, C)
        yp = (probs[ng_idx2] >= 0.5).astype(int)
        print(f"{w:>6} {C:>4} {(yt == yp).mean():>9.3f} {(yp[yt == 1] == 1).mean():>12.3f} {(yp[yt == 0] == 0).mean():>11.3f}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/135 [00:00<?, ?it/s]


weight    C  accuracy  scam recall  normal acc
    10    1     0.872        0.913       0.835
    10   10     0.872        0.893       0.853
    20    1     0.881        0.900       0.865
    20   10     0.878        0.900       0.859
    50    1     0.900        0.920       0.882
    50   10     0.891        0.927       0.859


In [ ]:
from sklearn.model_selection import train_test_split

b_train, b_hold = train_test_split(
    base_idx2, test_size=0.2, random_state=42, stratify=y2[base_idx2]
)

print(f"{'weight':>6} {'general accuracy':>17} {'scam recall':>12} {'normal acc':>11}")
for w in [1, 20, 50]:
    train_idx = np.concatenate([b_train, ng_idx2])
    weights = np.where(is_ng2[train_idx], float(w), 1.0)
    clf = LogisticRegression(max_iter=2000, C=1)
    clf.fit(X3[train_idx], y2[train_idx], sample_weight=weights)
    pred = clf.predict(X3[b_hold])
    yh = y2[b_hold]
    print(f"{w:>6} {(pred == yh).mean():>17.3f} {(pred[yh == 1] == 1).mean():>12.3f} {(pred[yh == 0] == 0).mean():>11.3f}")

weight  general accuracy  scam recall  normal acc
     1             0.980        0.978       0.983
    20             0.956        0.945       0.968
    50             0.944        0.925       0.963


In [ ]:
import joblib
from google.colab import files

overlap = set(test_df["message"]) & set(all2["message"])
print("Test messages also found in training data:", len(overlap))

weights = np.where(is_ng2, 20.0, 1.0)
final_clf = LogisticRegression(max_iter=2000, C=1)
final_clf.fit(X3, y2, sample_weight=weights)

X_test = embedder2.encode(test_df["message"].tolist(), batch_size=32)
y_test = (test_df["label"] == "scam").astype(int).values
test_probs = final_clf.predict_proba(X_test)[:, 1]
test_pred = (test_probs >= 0.5).astype(int)

tp = int(((test_pred == 1) & (y_test == 1)).sum())
fp = int(((test_pred == 1) & (y_test == 0)).sum())
fn = int(((test_pred == 0) & (y_test == 1)).sum())
tn = int(((test_pred == 0) & (y_test == 0)).sum())

print("\n=== LOCKED TEST SET (80 messages) ===")
print(f"Accuracy:       {(tp + tn) / 80:.1%}")
print(f"Scam recall:    {tp / (tp + fn):.1%}  ({tp} of {tp + fn} scams caught)")
print(f"Scam precision: {tp / (tp + fp):.1%}")
print(f"Normal acc:     {tn / (tn + fp):.1%}  ({fp} false alarms)")
print(f"Confusion: caught={tp}, missed={fn}, false alarms={fp}, correct normal={tn}")

print("\n--- Mistakes ---")
for m, a, p in zip(test_df["message"], y_test, test_probs):
    if (p >= 0.5) != bool(a):
        kind = "MISSED SCAM" if a == 1 else "FALSE ALARM"
        print(f"[{kind} | scam prob {p:.2f}] {m}")

joblib.dump(
    {"classifier": final_clf, "embedding_model": "all-mpnet-base-v2", "threshold": 0.5},
    "phishing_detector_v2.joblib",
)
files.download("phishing_detector_v2.joblib")

Test messages also found in training data: 0

=== LOCKED TEST SET (80 messages) ===
Accuracy:       91.2%
Scam recall:    87.5%  (35 of 40 scams caught)
Scam precision: 94.6%
Normal acc:     95.0%  (2 false alarms)
Confusion: caught=35, missed=5, false alarms=2, correct normal=38

--- Mistakes ---
[MISSED SCAM | scam prob 0.29] Please bro can you help me check my ATM card at home and copy the number behind it
[MISSED SCAM | scam prob 0.44] Your WAEC result is ready. Pay N2,500 and enter your email password to view.
[MISSED SCAM | scam prob 0.43] Update your bank app now to avoid losing your money: http://bit.ly/bank-update-ng
[MISSED SCAM | scam prob 0.42] Please do me a favour and tell me the last 4 digits and the CVV of your debit card, I want to pay your fees.
[MISSED SCAM | scam prob 0.35] Urgent! Your DSTV subscription expires today. Enter card details at dstv-renew-now.net.
[FALSE ALARM | scam prob 0.79] I need to change my phone number on my bank profile, who knows the process?


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
my_scams = [
    "we notice that its been a long time you use your bank account please use the link below to change your pin",
    "you are just been selected to participate in neolife resolution fill in your bvn and bank account to verify it",
    "you are been selected in a conference but you have to fill in your bank details for verification",
    "treat with urgency your account has ben freeze use the link below to set your account",
    "dont allow a little stipen to stop you from achieving your dream commit today with as little as #5000 to be selected by the federal government scholarship",
]

my_normals = [
    "congratulations,you have been selected for the cohort you can visit our website for more details",
    "after thorough selection  you where selected can you use the link to continue the next process",
    "use this link to continue you password reseting, but you can ignore if you did not ask for it",
    "congratulation you have been selected you can move on to the pying section using our correct platform",
    "good day mr ridwan, please continue your registration to be qualified for the opportunity attach to our platform"
]

msgs = my_scams + my_normals
labels = np.array([1] * len(my_scams) + [0] * len(my_normals))

emb = embedder2.encode(msgs, batch_size=32)
p = final_clf.predict_proba(emb)[:, 1]
pred = (p >= 0.5).astype(int)

for m, a, pp in zip(msgs, labels, p):
    verdict = "SCAM" if pp >= 0.5 else "normal"
    ok = "OK   " if (pp >= 0.5) == bool(a) else "WRONG"
    print(f"{ok} [{pp:.2f} -> {verdict}] {m[:100]}")

print(f"\nAccuracy: {(pred == labels).mean():.1%} on {len(msgs)} messages")
print(f"Scams caught: {int((pred[labels == 1] == 1).sum())} of {len(my_scams)}")
print(f"False alarms: {int((pred[labels == 0] == 1).sum())} of {len(my_normals)}")

OK    [0.94 -> SCAM] we notice that its been a long time you use your bank account please use the link below to change yo
OK    [0.99 -> SCAM] you are just been selected to participate in neolife resolution fill in your bvn and bank account to
WRONG [0.48 -> normal] you are been selected in a conference but you have to fill in your bank details for verification
OK    [0.90 -> SCAM] treat with urgency your account has ben freeze use the link below to set your account
WRONG [0.03 -> normal] dont allow a little stipen to stop you from achieving your dream commit today with as little as #500
WRONG [0.73 -> SCAM] congratulations,you have been selected for the cohort you can visit our website for more details
OK    [0.33 -> normal] after thorough selection  you where selected can you use the link to continue the next process
OK    [0.08 -> normal] use this link to continue you password reseting, but you can ignore if you did not ask for it
WRONG [0.91 -> SCAM] congratulation you have been se

In [ ]:
extra_normals = [
    "congratulations, you have been selected for the cohort 4 of the data science bootcamp, visit our website for more details",
    "Good day, you have been shortlisted for interview on monday 10am at our office in Akure, bring your CV and ID. no payment is required",
    "you have been selected as a class rep, please come to the department office tomorrow to sign",
    "hello your application for the internship has been received, we will get back to you within 2 weeks",
    "Congratulation, you have been selected for the scholarship interview, check your email for the date and venue",
    "pls use this link to reset ur password, ignore it if u did not request for it",
    "your registration for the tech summit is confirmed, see you on saturday at the hall",
    "Dear applicant, kindly log in to the official portal to complete your registration, no payment needed at this stage",
    "good morning mr ridwan, your course materials are ready, you can download from the student portal",
    "you have been added to the whatsapp group for the coding class, click the link to join",
    "the conference holds on friday, you have been selected to present your paper, please send the slides to my email",
    "your account password was changed successfully, if it wasnt you please contact support",
    "pls remind ur friends the registration closes tomorrow, go to the school website to register",
    "hi, you have been selected for the football team, training starts tomorrow 4pm",
    "your bank alert for N5,000 has been received, thanks for the payment",
    "congrats, you made the shortlist for the hackathon, team leads will send details tomorrow",
    "please continue your application on our platform, your progress was saved",
    "after the selection process we will contact the successful candidates by email, thanks for applying",
    "use this link to join the zoom class at 6pm today",
    "your scholarship result is out, visit the official website to check, never pay anyone for it",
    "good day sir, kindly confirm if you received my mail about the project",
    "i have been selected for the NYSC camp in ondo state, i go leave next week",
    "they selected me for the training, God is faithful",
    "your seat for the seminar is reserved, please come with your id card",
    "Welcome to the bootcamp! Your first class is on Tuesday. Join using the link in your email",
    "dear student you have been selected for the scholarship interview, bring your original documents, no fee is collected",
    "Congratulation on your selection! you can now proceed to the payment of school fees through the official school portal only",
    "pls the link i sent is the correct one, you can open it to continue the form",
    "you are invited to a free webinar on python on saturday, register with your email",
    "hello, your job application was successful, hr will call you on monday to discuss the next step",
]

extra_scams = [
    "we notice say u never use ur account for long time, use the link below to change ur pin or e go block",
    "you have been selected to participate in the national youth empowerment, fill your bvn and account number to verify",
    "congratulation you are selected for the oil company recruitment, send #10,000 for medical and your bank details",
    "your account has been frozen, kindly use the link below to unfreeze it with your pin and bvn",
    "dont let small money stop your dream, pay #5000 today to be selected for the federal government scholarship",
    "you have been selected for a free laptop programme, just fill in your bank details so we can verify you",
    "we notice unusual login on ur account, click the link and confirm ur card pin now",
    "you won the monthly draw of #500,000, send ur account number and bvn to receive it",
    "Commit just #3000 today to secure your slot in the federal government empowerment, limited slots",
    "your atm card is expire, click here to renew with ur pin, dont delay",
    "you are selected for an international conference in dubai, pay #20,000 processing fee and send your passport",
    "dear customer, ur bvn is not verified. verify now with the link below or ur account go be restricted",
    "treat with urgency, your account is on hold, use the link to release it and enter your password",
    "you have been picked for the cohort, to confirm your slot send your bank details and #2,500",
    "congratulation, selected for the tech grant of #1,000,000. fill ur bvn and account to claim",
    "pls help me with the code wey dem send to ur phone, na emergency",
    "ur loan don approve o, just pay #7,000 insurance first make dem send the money",
    "hello, your bank is upgrading. confirm your account details with the link below to avoid closure",
    "you have been selected for the visa lottery, pay #15,000 to the agent to continue",
    "dear user your sim go block today, click the link and put your nin to stop it",
    "your scholarship is ready but you must pay #4,000 clearance and send your atm card pin",
    "pls forward the otp wey enter ur phone, I send am by mistake",
    "you are among the 100 selected for the dangote scholarship, click the link and enter your bank details",
    "last warning: ur account will be deleted, click here and enter ur password to save am",
    "get selected for govt job by paying #6,000 to our agent, no exam no interview",
    "your payment of #3,500 failed, click the link and enter ur card number and pin to retry",
    "congratulations, your number was selected for free data, enter ur pin on the link to activate",
    "pls bro send me the number at the back of ur card, I wan send you money",
    "dear applicant, you have been admitted without exam. pay #50,000 acceptance fee into this account today",
    "we noticed ur bank app is old, update now with the link and login with ur pin",
]

user_df = pd.DataFrame({
    "message": my_scams + my_normals,
    "label": ["scam"] * len(my_scams) + ["ham"] * len(my_normals),
})
informal_df = pd.DataFrame({
    "message": extra_scams + extra_normals,
    "label": ["scam"] * len(extra_scams) + ["ham"] * len(extra_normals),
})

nigerian_train = pd.concat([nigerian_train, user_df, informal_df], ignore_index=True)
nigerian_train = nigerian_train.drop_duplicates(subset="message")
nigerian_train.to_csv("nigerian_train.csv", index=False)

print(nigerian_train.shape)
print(nigerian_train["label"].value_counts())

(330, 3)
label
ham     175
scam    155
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

# keep embeddings we already computed (must run before all2 is rebuilt)
lookup3 = dict(zip(all2["message"], X3))

# clean the leftover column and re-save
nigerian_train = nigerian_train[["message", "label"]]
nigerian_train.to_csv("nigerian_train.csv", index=False)

all2 = pd.concat([
    base.assign(source="base"),
    nigerian_train.assign(source="nigerian"),
    train_first60.assign(source="first60"),
], ignore_index=True).drop_duplicates(subset="message").reset_index(drop=True)

missing = [m for m in all2["message"] if m not in lookup3]
print("New messages to embed:", len(missing))
if missing:
    new_emb = embedder2.encode(missing, batch_size=32, show_progress_bar=True)
    for m, e in zip(missing, new_emb):
        lookup3[m] = e

X3 = np.vstack([lookup3[m] for m in all2["message"]])
y2 = (all2["label"] == "scam").astype(int).values
is_ng2 = (all2["source"] != "base").values
ng_idx2 = np.where(is_ng2)[0]
base_idx2 = np.where(~is_ng2)[0]
print("Nigerian-style messages:", len(ng_idx2))

print(f"\n--- Cross-validation on Nigerian-style messages ---")
print(f"{'weight':>6} {'accuracy':>9} {'scam recall':>12} {'normal acc':>11}")
yt = y2[ng_idx2]
for w in [20, 50]:
    probs = run_cv2(X3, w, 1)
    yp = (probs[ng_idx2] >= 0.5).astype(int)
    print(f"{w:>6} {(yt == yp).mean():>9.3f} {(yp[yt == 1] == 1).mean():>12.3f} {(yp[yt == 0] == 0).mean():>11.3f}")

print(f"\n--- Held-out general messages ---")
print(f"{'weight':>6} {'accuracy':>9} {'scam recall':>12} {'normal acc':>11}")
b_train, b_hold = train_test_split(base_idx2, test_size=0.2, random_state=42, stratify=y2[base_idx2])
for w in [20, 50]:
    train_idx = np.concatenate([b_train, ng_idx2])
    weights = np.where(is_ng2[train_idx], float(w), 1.0)
    clf = LogisticRegression(max_iter=2000, C=1)
    clf.fit(X3[train_idx], y2[train_idx], sample_weight=weights)
    pred = clf.predict(X3[b_hold])
    yh = y2[b_hold]
    print(f"{w:>6} {(pred == yh).mean():>9.3f} {(pred[yh == 1] == 1).mean():>12.3f} {(pred[yh == 0] == 0).mean():>11.3f}")

New messages to embed: 70


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Nigerian-style messages: 390

--- Cross-validation on Nigerian-style messages ---
weight  accuracy  scam recall  normal acc
    20     0.905        0.914       0.898
    50     0.900        0.903       0.898

--- Held-out general messages ---
weight  accuracy  scam recall  normal acc
    20     0.951        0.938       0.965
    50     0.936        0.920       0.953


In [2]:
!pip install -q sentence-transformers

from google.colab import files
uploaded = files.upload()
print(list(uploaded.keys()))

Saving nigerian_train (1).csv to nigerian_train (1) (1).csv
Saving train_first60.csv to train_first60.csv
Saving locked_test_80.csv to locked_test_80.csv
Saving base_reduced.csv to base_reduced.csv
['nigerian_train (1) (1).csv', 'train_first60.csv', 'locked_test_80.csv', 'base_reduced.csv']


In [3]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression

def load_longest(prefix):
    candidates = [pd.read_csv(f) for f in uploaded if f.startswith(prefix)]
    return max(candidates, key=len)

base = load_longest("base_reduced")
nigerian_train = load_longest("nigerian_train")[["message", "label"]]
train_first60 = load_longest("train_first60")
test_df = load_longest("locked_test_80")
print("base:", len(base), "| nigerian_train:", len(nigerian_train),
      "| first60:", len(train_first60), "| locked test:", len(test_df))

all2 = pd.concat([
    base.assign(source="base"),
    nigerian_train.assign(source="nigerian"),
    train_first60.assign(source="first60"),
], ignore_index=True).drop_duplicates(subset="message").reset_index(drop=True)

embedder2 = SentenceTransformer("all-mpnet-base-v2")
embedder2.max_seq_length = 128
X3 = embedder2.encode(all2["message"].tolist(), batch_size=32, show_progress_bar=True)

y2 = (all2["label"] == "scam").astype(int).values
is_ng2 = (all2["source"] != "base").values

weights = np.where(is_ng2, 20.0, 1.0)
final_clf2 = LogisticRegression(max_iter=2000, C=1)
final_clf2.fit(X3, y2, sample_weight=weights)
print("Model trained on", len(all2), "messages")

base: 4000 | nigerian_train: 330 | first60: 60 | locked test: 80


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/138 [00:00<?, ?it/s]

Model trained on 4390 messages


In [4]:
r2_normals = [
    "Hi John, our meeting has been moved to 10 AM tomorrow. Please confirm your availability.",
    "Your university timetable for the new semester is now available on the student portal.",
    "Thank you for shopping with us. Your order has been confirmed and will be delivered within three working days.",
    "Hello, please find attached the lecture notes from today's class.",
    "Your electricity bill for November is ready. You can view it through your usual account.",
    "Good morning, the department seminar will hold on Friday at 12 PM.",
    "Your password was successfully changed. If you did not make this change, contact our support team.",
    "Reminder: Your appointment is scheduled for Monday at 9 AM.",
    "The project document has been updated. Please review the latest version before our next meeting.",
    "Your monthly bank statement is available through your mobile banking application.",
    "Hello, I have received your assignment. Thank you for submitting it before the deadline.",
    "Your subscription will renew on December 5. You can manage your preferences in your account settings.",
    "The examination schedule has been published on the faculty notice board.",
    "Thank you for registering for the cybersecurity awareness workshop. We look forward to seeing you.",
    "Your parcel has been delivered successfully. Thank you for choosing our delivery service.",
]

r2_scams = [
    "URGENT! Your bank account will be suspended today. Verify your details immediately using this link.",
    "Congratulations! You have won ₦500,000. Click here to claim your prize now.",
    "Your email account has been compromised. Send your password to our security team immediately.",
    "Your ATM card has been blocked. Click the link below to restore access.",
    "FINAL WARNING: Your account will be permanently deleted unless you verify your identity within 30 minutes.",
    "You have an unpaid delivery fee. Pay immediately using this unfamiliar payment link to receive your package.",
    "Dear customer, we detected suspicious activity. Enter your banking PIN to prevent account closure.",
    "Your university scholarship has been approved. Pay a processing fee immediately to receive your funds.",
    "Your social media account will be disabled. Confirm your login credentials through this link.",
    "You have been selected for an exclusive cash reward. Provide your bank details to receive payment.",
    "Your cloud storage is full. Download this attachment immediately to prevent permanent data loss.",
    "Your mobile banking access has expired. Reply with your password and verification code to reactivate it.",
    "We are updating our security system. Click this unofficial link and enter your login information.",
    "Your tax refund is waiting. Submit your bank account details to receive your money today.",
    "Someone attempted to access your account. Download the attached security update and run it immediately.",
]

msgs = r2_scams + r2_normals
labels = np.array([1] * len(r2_scams) + [0] * len(r2_normals))

emb = embedder2.encode(msgs, batch_size=32)
p = final_clf2.predict_proba(emb)[:, 1]
pred = (p >= 0.5).astype(int)

for m, a, pp in zip(msgs, labels, p):
    verdict = "SCAM" if pp >= 0.5 else "normal"
    ok = "OK   " if (pp >= 0.5) == bool(a) else "WRONG"
    print(f"{ok} [{pp:.2f} -> {verdict}] {m[:100]}")

print(f"\nAccuracy: {(pred == labels).mean():.1%} on {len(msgs)} messages")
print(f"Scams caught: {int((pred[labels == 1] == 1).sum())} of {len(r2_scams)}")
print(f"False alarms: {int((pred[labels == 0] == 1).sum())} of {len(r2_normals)}")

OK    [0.99 -> SCAM] URGENT! Your bank account will be suspended today. Verify your details immediately using this link.
OK    [0.97 -> SCAM] Congratulations! You have won ₦500,000. Click here to claim your prize now.
OK    [0.89 -> SCAM] Your email account has been compromised. Send your password to our security team immediately.
OK    [0.98 -> SCAM] Your ATM card has been blocked. Click the link below to restore access.
WRONG [0.50 -> normal] FINAL WARNING: Your account will be permanently deleted unless you verify your identity within 30 mi
OK    [0.87 -> SCAM] You have an unpaid delivery fee. Pay immediately using this unfamiliar payment link to receive your 
OK    [0.93 -> SCAM] Dear customer, we detected suspicious activity. Enter your banking PIN to prevent account closure.
OK    [0.74 -> SCAM] Your university scholarship has been approved. Pay a processing fee immediately to receive your fund
OK    [0.69 -> SCAM] Your social media account will be disabled. Confirm your login cr

In [5]:
import joblib
from google.colab import files

joblib.dump(
    {"classifier": final_clf2, "embedding_model": "all-mpnet-base-v2", "threshold": 0.5},
    "phishing_detector_v2_1.joblib",
)
files.download("phishing_detector_v2_1.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
import sklearn, sentence_transformers, numpy, joblib, torch

print("scikit-learn", sklearn.__version__)
print("sentence-transformers", sentence_transformers.__version__)
print("numpy", numpy.__version__)
print("joblib", joblib.__version__)
print("torch", torch.__version__)

scikit-learn 1.6.1
sentence-transformers 5.7.0
numpy 2.4.6
joblib 1.6.0
torch 2.14.1+cpu
